In [2]:
%load_ext autoreload
%autoreload 2

In [9]:
import os
import sys
import json
import librosa
from IPython.display import Audio

import torch
import peft
from transformers import AutoModelForSeq2SeqLM, AutoProcessor

from AADLLM.trainer import load_lora
from AADLLM.prepare_tokenizer import prepare
from AADLLM.AADLLM import AADLLM, input_template

os.environ['TOKENIZERS_PARALLELISM'] = 'True'

ckpt_path = None
assert ckpt_path != None
xvect_centroids_path = 'AADLLM/xvect_K=8.pt'

## Prepare models

In [4]:
qwen2 = AutoModelForSeq2SeqLM.from_pretrained(
    pretrained_model_name_or_path="Qwen/Qwen2-Audio-7B-Instruct",
    device_map="cpu",
    trust_remote_code=True,
)

print('Initialize original Qwen2-Audio')

torch.cuda.empty_cache()

peft_config = peft.LoraConfig(
    r=512,
    lora_alpha=512,
    target_modules=[
        "k_proj", "v_proj", "q_proj","out_proj", "o_proj",
        "fc1", "fc2", "value", "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
)

lora_qwen2 = peft.get_peft_model(
    model=qwen2,
    peft_config=peft_config,
)

print('Add LoRA to Qwen2-Audio')

torch.cuda.empty_cache()

# This will add two special tokens to Qwen2's tokenizer
tokenizer = prepare(
    spk_hint=True,
    model='Qwen/Qwen2-Audio-7B-Instruct'
)

extra_123_id = tokenizer('<|extra_123|>').input_ids[0]
print('<SEP> (<|extra_123|>) id:', extra_123_id)
extra_124_id = tokenizer('<|extra_124|>').input_ids[0]
print('<hint> (<|extra_124|>) id:', extra_124_id)


processor = AutoProcessor.from_pretrained(
    pretrained_model_name_or_path="Qwen/Qwen2-Audio-7B-Instruct"
)

processor.tokenizer = tokenizer

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Initialize original Qwen2-Audio
Add LoRA to Qwen2-Audio
<SEP> (<|extra_123|>) id: 154931
<hint> (<|extra_124|>) id: 154932


## Build AAD-LLM

In [7]:
model = AADLLM(
    qwen=lora_qwen2,
    spk_hint=True,
    spks_vects=False,
    same_proj_hint_vects=True,
    hint_tok_id=extra_124_id,
    spks_tok_id=extra_123_id,
    spk_dim=512 # dim of ECAPA-xvector
)

print('Build AAD-LLM based on Qwen2-Audio')

Use a linear layer to project attended cluster onto the LLM input space.
Don't append speaker vectors.
No additional layer between every LLM layer.
Build AAD-LLM based on Qwen2-Audio


## Load weights

In [10]:
load_lora(model.qwen, os.path.join(ckpt_path, 'lora.pt'))
model.hinter.load_state_dict(
    torch.load(
        os.path.join(ckpt_path, 'hinter.pt')
    )   
)

# This is the K-Menas centroids (K=8) of speaker identity vectors
# We use one of this to condition LLM
xvect_centroids = torch.load(xvect_centroids_path).cuda()
print(xvect_centroids.shape, xvect_centroids.device)

torch.Size([8, 512]) cuda:0


In [11]:
model = model.cuda()
model.hinter = model.hinter.cuda()
model.device = torch.device('cuda')
torch.cuda.empty_cache()

## Load sample data

In [20]:
tests = json.load(open('samples/test_subset.json', 'r'))

i = 1
x = tests[i]

In [22]:
# Mixture = speech1 + speech2 + noise
mix_path = x['path']

# Clean speech 1 and speech 2
src1_path = x['speeches'][0]['path']
src2_path = x['speeches'][1]['path']

# Separated speeches from any speech separation model (Mamba-TasNet in our case)
# In random order (sep1 correponds to either src1 or src2, we don't know)
sep1_path = x['separated_paths'][0]
sep2_path = x['separated_paths'][1]

# x-vector label of the attended speaker (either speaker 1 or speaker 2)
# In the paper (https://arxiv.org/abs/2502.16794), this is decoded from the brain by a small RNN
# Otherwise, it can come from a reference speaker
# For now, we just use the ground-truth x-vector label
attn_label = int(x['speeches'][1]['xvect8'])
attn_xvect = xvect_centroids[attn_label]

In [23]:
Audio(mix_path, rate=16000)

## Inference

In [24]:
question = 'Transcribe the attended talker.'

In [25]:
## Prepare inputs to AAD-LLM

prompt = input_template.format(
    question=question
)

wavs = [librosa.load(sep1_path, sr=16000)[0], librosa.load(sep2_path, sr=16000)[0]]

# Input to LLM
inputs = processor(text=prompt, audios=wavs, sampling_rate=16000, return_tensors='pt', padding=False)

# Qwen2Audio fixed expected_seq_length to be 3000
# but we don't need padding, so we remove here
audio_features = inputs['input_features']
audio_masks = inputs['feature_attention_mask']
M = audio_masks.sum(dim=1).long().max()
audio_features = audio_features[:, :, :M]
audio_masks = audio_masks[:, :M] 
inputs['input_features'] = audio_features
inputs['feature_attention_mask'] = audio_masks

# Add attended speaker's x-xvector
inputs['attn_cluster'] = attn_xvect

for k, v in inputs.items():
    inputs[k] = v.cuda()
    print(k, inputs[k].shape, inputs[k].device)

input_ids torch.Size([1, 272]) cuda:0
attention_mask torch.Size([1, 272]) cuda:0
input_features torch.Size([2, 128, 447]) cuda:0
feature_attention_mask torch.Size([2, 447]) cuda:0
attn_cluster torch.Size([512]) cuda:0


/tmp/ipykernel_100633/1688761876.py:10: FutureWarning: `audios` is deprecated and will be removed in version 4.54.0 for `Qwen2AudioProcessor.__call__`. Use `audio` instead.
  inputs = processor(text=prompt, audios=wavs, sampling_rate=16000, return_tensors='pt', padding=False)


In [26]:
torch.backends.cudnn.enabled = False

with torch.inference_mode():
    generate_ids = model.generate(inputs, max_length=1024)
    generate_ids = generate_ids[:, inputs.input_ids.size(1):]
    response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

In [27]:
print(question)
print(response)

Transcribe the attended talker.
Attention: 7; Spk1: 1; Spk2: 7; The girl was still seated on the chair, but she had changed her attitude.


## Appendix: compute x-vector and x-vector label

In [ ]:
import torch
import librosa
from speechbrain.inference.speaker import EncoderClassifier

xmodel = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb",
    savedir="xvect/spkrec-xvect-voxceleb",
    run_opts={"device": "cuda"},
)

def compute_spkemb(path):
    wav, sr = librosa.load(path, sr=16000)
    wav = torch.tensor(wav, dtype=torch.float32).cuda()
    emb = xmodel.encode_batch(wav.unsqueeze(0)).squeeze(0).squeeze(0)
    return emb

xvect = compute_spkemb(src1_path)

# Find the cloest centroid from xvect_centroids of shape [8, 512]
dists = torch.cdist(xvect.unsqueeze(0), xvect_centroids) # [1, 8]
xvect8_label = torch.argmin(dists, dim=1).item()